## Prerequisito


| Concepto | Que es | Ejemplo |
|----------|--------|---------|
| **Surrogate Key (SK)** | Clave artificial generada por el DW (IDENTITY). Inmutable, para JOINs | `zona_id = 42` en dim_zona |
| **Business Key (BK)** | Clave natural del negocio. Para matching en MERGE e identificar la entidad | `partido + region` en dim_zona |

En SCD2: mismo BK puede tener multiples SKs (una por version). El SK identifica la version, el BK identifica la entidad.

In [0]:
%sql
SELECT COUNT(*) AS total_registros 
FROM propiedades.silver.propiedades;

In [0]:
%sql
DESCRIBE TABLE propiedades.silver.propiedades;

---
# MODULO 1: Facts vs Dims - Clasificacion, Conceptos y SCDs

---

## 1.1: Clasificar campos como Fact o Dimension

**Objetivo:** Aplicar el criterio de cardinalidad y tipo de dato para separar metricas de atributos descriptivos.

Ejecuta una query que calcule `COUNT(DISTINCT campo)` para los campos clave. Despues, clasifica cada uno.

In [0]:
%sql
SELECT  COUNT(DISTINCT partido) AS dist_partido,
        COUNT(DISTINCT tipo_operacion) AS dist_tipo_op,
        COUNT(DISTINCT estado) AS dist_estado,
        COUNT(DISTINCT moneda) AS dist_moneda,
        COUNT(DISTINCT precio) AS dist_precio,
        COUNT(DISTINCT metros_cuadrados_totales) AS dist_m2,
        COUNT(DISTINCT url) AS dist_url,
        COUNT(*) AS total_filas

FROM propiedades.silver.propiedades;

### 1.1: Clasificacion

| Campo | Cardinalidad | Clasificacion | Justificacion |
|-------|-------------|---------------|---------------|
| `partido` | Baja (~36) | **Dimension** | Categorico, baja cardinalidad, agrupa registros |
| `tipo_operacion` | Muy baja (~9) | **Dimension** | Categorico, pocas categorias |
| `estado` | Baja (~20) | **Dimension** | Categorico, describe condicion del inmueble |
| `moneda` | Muy baja (~3-4) | **Degenerate dim / Dimension** | Tan pocos valores que puede ir en la fact o en una dim minima |
| `precio` | Alta | **Metrica (Fact)** | Numerico, se suma/promedia |
| `metros_cuadrados_totales` | Alta | **Metrica (Fact)** | Numerico, se suma/promedia |
| `url` | Casi unica por fila | **Degenerate Dimension** | Cardinalidad ~ total de filas, no justifica tabla propia, pero sirve como trazabilidad |

`url` es una **degenerate dimension**: vive en la fact table porque tiene cardinalidad altisima (casi 1:1 con las filas) y no tiene atributos descriptivos propios. Sirve para trazar cada registro a la publicacion original.

### Moneda necesita una nueva Dimension? 

**Con 3-4 valores sin atributos extras:** Degenerate dimension. No tiene sentido crear una tabla, un schema, un JOIN para 3 valores que son solo codigos. Va directo en la fact como `moneda STRING`.

**Si en el futuro se agregan 20 monedas con metadatos** (nombre completo, simbolo, pais de origen, tipo de cambio base): conviene crear `dim_moneda` porque ahora la moneda tiene **atributos descriptivos propios** que enriquecen el analisis.

**Criterio:** No es solo cardinalidad. La pregunta clave es: necesito atributos descriptivos extras sobre esta entidad? Si la respuesta es si -> dimension. Si es solo un codigo plano que no cambia -> degenerate dimension.

### 1.4

| Caso | SCD Type | Justificacion |
|------|----------|---------------|
| (a) Corregir typo "capital-fedral" -> "capital-federal" | **SCD Type 1** | Es una correccion de error, no un cambio de negocio. No necesitamos saber que alguna vez estuvo mal escrito. Sobrescribimos. |
| (b) Partido cambia de region (GBA Norte -> GBA Oeste), necesitas historico | **SCD Type 2** | Necesitas saber a que region pertenecia en cada momento para reportes historicos. Se crea nueva version con `valid_from`/`valid_to`. |
| (c) Barrio cambia de nombre, solo necesitas el anterior | **SCD Type 3** | Solo necesitas "antes y despues", no todo el historial. Se agrega columna `nombre_anterior` y se actualiza en la misma fila. |
| (d) Reclasificar tipo de operacion, reportes viejos deben reflejar el nuevo nombre | **SCD Type 1** | Si los reportes viejos deben mostrar el nombre nuevo, entonces el pasado se "reescribe". No me importa preservar el valor anterior. |

**Regla rapida:** SCD1 = no me importa el pasado. ** SCD2 = necesito todas las versiones.  ** SCD3 = solo necesito "antes y despues".

---
## 1.5: SCD Type 1 - Overwrite con MERGE

**Objetivo:** Implementar una correccion que sobreescriba el valor anterior sin guardar historico.
Actualice los campos descriptivos si tienen cambios

Sobre `propiedades.gold.dim_zona`, crear una vista staging con datos corregidos y aplicar MERGE.

In [0]:
%sql
-- Paso 1: Vista staging con correcciones
CREATE OR REPLACE TEMP VIEW staging_correcciones 
AS
SELECT  partido, 
        region, 
        'Gran Buenos Aires' AS ciudad

FROM    propiedades.gold.dim_zona

WHERE   region = 'gba norte' 
AND     ciudad = 'GBA';